In [0]:
-- verificacion tabla silver completa
SELECT
  COUNT(*)
FROM bootcamp_de_valentin.silver.propiedades_silver;

DESCRIBE TABLE bootcamp_de_valentin.silver.propiedades_silver;

# **Modulo 1: Modelado Gold - Star Schema**

## Tablas Dimensionales


In [0]:
-- Ejercicio 1.1 
-- Crear tablas de dimensiones
-- dim_zona
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_zona (
  zona_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  partido STRING NOT NULL,
  region STRING NOT NULL,
  ciudad STRING,
  provincia STRING DEFAULT 'Buenos Aires',
  pais STRING DEFAULT 'Argentina',
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES( 'delta.feature.allowColumnDefaults' = 'supported' )
COMMENT 'Dimension de zonas - Star Schema';

INSERT INTO bootcamp_de_valentin.gold.dim_zona (partido, region, ciudad, provincia, pais)
SELECT DISTINCT
  partido,
  region,
  CASE
    WHEN region = 'capital federal' THEN 'CABA'
    ELSE 'GBA'
  END AS ciudad,
  'Buenos Aires' AS provincia,
  'Argentina' AS pais
FROM bootcamp_de_valentin.silver.propiedades_silver
WHERE partido IS NOT NULL
ORDER BY partido;

In [0]:
-- dim_tipo_operacion
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_tipo_operacion(
  tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  tipo_operacion STRING NOT NULL,
  moneda STRING NOT NULL,
  categoria STRING,
  descripcion STRING,
  _created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES('delta.feature.allowColumnDefaults'= 'supported')
COMMENT 'Dimension Tipo Operacion + moneda - Star Schema' ;

INSERT INTO bootcamp_de_valentin.gold.dim_tipo_operacion (tipo_operacion, moneda, categoria, descripcion)
SELECT DISTINCT
  tipo_operacion,
  moneda,
  CASE
    WHEN tipo_operacion = 'alquiler' THEN 'residencial'
    WHEN tipo_operacion = 'venta' THEN 'residencial'
    WHEN tipo_operacion = 'alquiler_temporario' THEN 'temporal'
    ELSE 'otro'
  END AS categoria,
  CASE
    WHEN tipo_operacion = 'alquiler' THEN 'Alquiler residencial'
    WHEN tipo_operacion = 'venta' THEN 'Venta de propiedad'
    WHEN tipo_operacion = 'alquiler_temporario' THEN 'Alquiler temporal'
    ELSE 'Otro tipo'
  END AS descripcion
FROM bootcamp_de_valentin.silver.propiedades_silver
WHERE 
  tipo_operacion IS NOT NULL
  AND moneda IS NOT NULL
ORDER BY tipo_operacion, moneda;

In [0]:
-- dim_tiempo
DROP TABLE bootcamp_de_valentin.gold.dim_tiempo;

CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_tiempo (
  fecha_id BIGINT,
  fecha DATE NOT NULL,
  anio INT,
  anio_mes STRING,
  anio_mes_num INT,
  mes STRING,
  mes_num INT,
  trimestre INT,
  dia_semana STRING,
  dia_semana_num INT,
  es_fin_de_semana BOOLEAN,
  _created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimension tiempo - Star Schema';

INSERT INTO bootcamp_de_valentin.gold.dim_tiempo (fecha_id, fecha, anio, anio_mes, anio_mes_num, mes, mes_num, trimestre, dia_semana, dia_semana_num, es_fin_de_semana)
SELECT DISTINCT
  CAST( DATE_FORMAT(fecha_publicacion, 'yyyyMMdd') AS BIGINT ) AS fecha_id,
  CAST( fecha_publicacion AS DATE) AS fecha,
  YEAR(fecha_publicacion) AS anio,
  DATE_FORMAT(fecha_publicacion, 'yy-MMM') AS anio_mes,
  CAST( DATE_FORMAT(fecha_publicacion, 'yyMM') AS INT) AS anio_mes_num,
  DATE_FORMAT(fecha_publicacion, 'MMM') AS mes,
  MONTH(fecha_publicacion) AS mes_num,
  QUARTER(fecha_publicacion) AS trimestre,
  DATE_FORMAT(fecha_publicacion, 'EEE') AS dia_semana,
  EXTRACT(DAYOFWEEK_ISO FROM fecha_publicacion) AS dia_semana_num,
  CASE
    WHEN EXTRACT(DAYOFWEEK_ISO FROM fecha_publicacion) IN (6,7) THEN TRUE
    ELSE FALSE
  END AS es_fin_de_semana
FROM bootcamp_de_valentin.silver.propiedades_silver
WHERE fecha_publicacion IS NOT NULL
ORDER BY fecha;

In [0]:
-- dim_tiempo_2

-- CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_tiempo (
--   fecha_id BIGINT,
--   fecha DATE NOT NULL,
--   anio INT,
--   anio_mes STRING,
--   anio_mes_num INT,
--   mes STRING,
--   mes_num INT,
--   trimestre INT,
--   dia_semana STRING,
--   dia_semana_num INT,
--   es_fin_de_semana BOOLEAN,
--   _created_at TIMESTAMP DEFAULT current_timestamp()
-- )
-- USING DELTA
-- TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
-- COMMENT 'Dimension tiempo - Star Schema';

-- INSERT INTO bootcamp_de_valentin.gold.dim_tiempo2 (fecha_id, fecha, anio, anio_mes, anio_mes_num, mes, mes_num, trimestre, dia_semana, dia_semana_num, es_fin_de_semana)


In [0]:
-- dim_caracteristicas
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_caracteristicas (
  caracteristica_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  estado STRING NOT NULL,
  cochera BOOLEAN,
  _created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Junk Dimension Caracteristicas (estado + cochera) - Star Schema';

INSERT INTO bootcamp_de_valentin.gold.dim_caracteristicas (estado, cochera)
SELECT DISTINCT
  COALESCE(estado, 'sin especificar') AS estado,
  COALESCE( cochera, false ) AS cochera
FROM bootcamp_de_valentin.silver.propiedades_silver;


In [0]:
-- dim_orientacion
CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.dim_orientacion (
  orientacion_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
  orientacion STRING NOT NULL,
  tipo_orientacion STRING,
  _created_at TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Dimension Orientacion - Star Schema';

INSERT INTO bootcamp_de_valentin.gold.dim_orientacion (orientacion, tipo_orientacion)
SELECT DISTINCT
  COALESCE(orientacion, 'sin especificar') as orientacion,
  CASE 
      WHEN LOWER(orientacion) LIKE '%norte%' THEN 'norte'
      WHEN LOWER(orientacion) LIKE '%sur%' THEN 'sur'
      WHEN LOWER(orientacion) LIKE '%este%' THEN 'este'
      WHEN LOWER(orientacion) LIKE '%oeste%' THEN 'oeste'
      ELSE 'sin especificar'
  END as tipo_orientacion
FROM bootcamp_de_valentin.silver.propiedades_silver
ORDER BY orientacion;

In [0]:
-- Verificacion de dimensiones creadas
SELECT * FROM bootcamp_de_valentin.gold.dim_zona

In [0]:
SELECT * FROM bootcamp_de_valentin.gold.dim_tipo_operacion

In [0]:
SELECT * FROM bootcamp_de_valentin.gold.dim_tiempo

In [0]:
SELECT * FROM bootcamp_de_valentin.gold.dim_caracteristicas

In [0]:
SELECT * FROM bootcamp_de_valentin.gold.dim_orientacion

## Tabla de Hechos

In [0]:

DROP TABLE IF EXISTS bootcamp_de_valentin.gold.fact_propiedades;

CREATE TABLE IF NOT EXISTS bootcamp_de_valentin.gold.fact_propiedades (
  -- Row Hash como PK
  row_hash STRING NOT NULL,
  -- Foreing Keys
  zona_id BIGINT,
  tipo_operacion_id BIGINT,
  fecha_id BIGINT, 
  caracteristicas_id BIGINT,
  orientacion_id BIGINT,
  -- Degenerated Dimension
  url STRING,
  -- Metricas
  precio DECIMAL (15,2),
  expensas DECIMAL (15,2),
  precio_m2 DECIMAL (15,2),
  m2_totales DECIMAL (15,2),
  m2_cubiertos DECIMAL (15,2),
  ambientes INT,
  -- Fecha de actualizacion
  _refresh_timestamp TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de Hechos Alquiler Propiedades - Star Schema - PK = row_hash (MD5 de url + precio)';

INSERT INTO bootcamp_de_valentin.gold.fact_propiedades (
  row_hash, 
  zona_id, tipo_operacion_id, fecha_id, caracteristicas_id, orientacion_id,
  precio, expensas, precio_m2, m2_totales, m2_cubiertos, ambientes, 
  url
)
SELECT
  -- pk
  MD5( CONCAT_WS('|', precio, url)) AS row_hash,
  -- fks
  dz.zona_id,
  dtp.tipo_operacion_id,
  dt.fecha_id,
  dc.caracteristica_id,
  do.orientacion_id,
  -- metricas
  ps.precio,
  ps.expensas,
  ps.precio_por_m2,
  ps.m2_totales,
  ps.m2_cubiertos,
  ps.ambientes,
  -- degenerated dimension
  ps.url
FROM bootcamp_de_valentin.silver.propiedades_silver ps
LEFT JOIN bootcamp_de_valentin.gold.dim_zona dz
  ON dz.partido = ps.partido
  AND dz.region = ps.region
LEFT JOIN bootcamp_de_valentin.gold.dim_tipo_operacion dtp
  ON dtp.tipo_operacion = ps.tipo_operacion
  AND dtp.moneda = ps.moneda
LEFT JOIN bootcamp_de_valentin.gold.dim_tiempo dt
  ON dt.fecha = ps.fecha_publicacion
LEFT JOIN bootcamp_de_valentin.gold.dim_caracteristicas dc
  ON dc.estado = COALESCE( ps.estado, 'sin especificar')
  AND dc.cochera = COALESCE( ps.cochera, false)
LEFT JOIN bootcamp_de_valentin.gold.dim_orientacion do
  ON do.orientacion = COALESCE( ps.orientacion, 'sin especifica');

In [0]:
-- chequeo tabla fact
SELECT
  COUNT(*) total_Registros,
  COUNT(DISTINCT row_hash) hashes_distintos
FROM bootcamp_de_valentin.gold.fact_propiedades

# **Modulo 1: Modelado Gold - Snowflake, OBT, Galaxy**